In [153]:
import torch.nn as nn
import torch
import pandas as pd
import json
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

In [154]:
import torch
import torch.nn as nn


class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 1,
        hidden_dim: int = 64,
        latent_dim: int = 20,
        num_layers: int = 1,
    ):
        super().__init__()

        # ---------- Encoder ---------- #
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        # ---------- Bottleneck ---------- #
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)

        # ---------- Decoder ---------- #
        self.decoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        # ---------- Output ---------- #
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(f"Expected (B,T,F), got {x.shape}")

        _, (h, _) = self.encoder(x)

        h = h[-1]

        latent = self.to_latent(h)
        hidden_dec = self.from_latent(latent)

        # FIX 1: decoder input should carry sequence info
        dec_in = x

        h0 = hidden_dec.unsqueeze(0).repeat(self.decoder.num_layers, 1, 1)
        c0 = torch.zeros_like(h0)

        out, _ = self.decoder(dec_in, (h0, c0))
        out = self.output_layer(out)

        return out

In [155]:
# ---------- Load time series ---------- #
def load_series(csv_path: str):
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df


# ---------- Load anomaly windows ---------- #
def load_windows(json_path: str, file_key: str):
    with open(json_path, "r") as f:
        data = json.load(f)
    return data[file_key]


# ---------- Build point labels ---------- #
def build_labels(df, windows):
    labels = np.zeros(len(df))

    for start, end in windows:
        start = pd.to_datetime(start)
        end = pd.to_datetime(end)

        mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
        labels[mask] = 1

    df["label"] = labels
    return df


In [156]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset


# ---------- Windowing ---------- #
def make_windows_train(values, labels, seq_len: int):
    values = np.asarray(values).reshape(-1, 1)
    labels = np.asarray(labels)

    n_windows = len(values) - seq_len + 1

    x_windows = np.empty((n_windows, seq_len, 1), dtype=np.float32)
    y_windows = np.empty((n_windows,), dtype=np.float32)

    for i in range(n_windows):
        x_windows[i] = values[i:i + seq_len]
        y_windows[i] = np.max(labels[i:i + seq_len])

    return x_windows, y_windows


# ---------- Training ---------- #
def train_model(model, train_x, lr=1e-3, epochs=10, batch_size=64, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()

    x = torch.tensor(train_x, dtype=torch.float32)

    if x.ndim != 3:
        raise ValueError(f"Invalid input shape: {x.shape}")

    dataset = TensorDataset(x)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device == "cuda")
    )

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0

        for (batch,) in loader:

            batch = batch.to(device)

            pred = model(batch)

            # FIX: same metric as inference
            loss = torch.mean((pred - batch) ** 2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        print(f"epoch {epoch+1}/{epochs} | loss {total_loss / max(n_batches,1):.6f}")

    return model

In [182]:
import numpy as np
import torch


# ---------- Windowing ---------- #
def make_windows_inference(values, seq_len: int):
    values = np.asarray(values).reshape(-1, 1)

    x_windows = []

    for i in range(len(values) - seq_len + 1):
        x_windows.append(values[i:i + seq_len])

    return np.stack(x_windows).astype(np.float32)


# ---------- Inference ---------- #
def run_inference(model, values, seq_len: int, device=None, batch_size=256):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model.eval()

    x = make_windows_inference(values, seq_len)
    x = torch.tensor(x, dtype=torch.float32)

    if x.ndim != 3:
        raise ValueError(f"Expected (B,T,F), got {x.shape}")

    dataset = torch.utils.data.TensorDataset(x)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)

    errors = []
    recons = []

    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)

            recon = model(batch)

            err = torch.mean((recon - batch) ** 2, dim=(1, 2))

            errors.append(err.cpu())
            recons.append(recon.cpu())

    errors = torch.cat(errors).numpy()
    recons = torch.cat(recons).numpy()

    return errors, recons


# ---------- Threshold (FIXED ONLY) ---------- #
def compute_threshold(train_errors):
    train_errors = np.asarray(train_errors)

    # FIX: robust calibration instead of unstable mean+std
    return np.percentile(train_errors, 95)


# ---------- Detection ---------- #
def detect_anomalies(errors, threshold):
    return (np.asarray(errors) > threshold).astype(np.int32)


# ---------- Pipeline ---------- #
def inference_pipeline(model, train_values, test_values, seq_len: int):
    train_err, _ = run_inference(model, train_values, seq_len)

    threshold = compute_threshold(train_err)

    test_err, _ = run_inference(model, test_values, seq_len)

    preds = detect_anomalies(test_err, threshold)

    return train_err, test_err, threshold, preds

In [165]:
import numpy as np
import torch


# ---- config ---- #
SEQ_LEN = 30
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


# ---- load data ---- #
df = load_series("../assets/data/art_daily_flatmiddle.csv")

windows = load_windows(
    json_path="../assets/labels/combined_windows.json",
    file_key="artificialWithAnomaly/art_daily_flatmiddle.csv"
)

df = build_labels(df, windows)

values = df["value"].values.astype(np.float32)
labels = df["label"].values.astype(np.float32)


# ---- windowing ---- #
x, y = make_windows_train(values, labels, SEQ_LEN)

# ---- TRUE ANOMALY STATS ---- #
true_anomaly_ratio = np.mean(y == 1)
print(f"true_anomaly_ratio: {true_anomaly_ratio:.6f} ({true_anomaly_ratio*100:.2f}%)")
print(f"windows: {len(y)}")
print(f"anomaly_windows: {np.sum(y == 1)}")
print(f"normal_windows: {np.sum(y == 0)}")


# ---- NORMAL ONLY TRAIN SET ---- #
train_x = x[y == 0]


# ---- model ---- #
model = LSTMAutoencoder(input_dim=1, hidden_dim=128, latent_dim=32)


# ---- train ---- #
model = train_model(
    model=model,
    train_x=train_x,
    epochs=50,
    batch_size=64,
    device=device
)


# ---- save model ---- #
torch.save(model.state_dict(), "lstm_ae_model.pt")
print("model saved")

device: cuda
true_anomaly_ratio: 0.107919 (10.79%)
windows: 4003
anomaly_windows: 432
normal_windows: 3571
epoch 1/50 | loss 2176.051494
epoch 2/50 | loss 1760.550011
epoch 3/50 | loss 1460.944397
epoch 4/50 | loss 1244.705597
epoch 5/50 | loss 1061.380144
epoch 6/50 | loss 890.185852
epoch 7/50 | loss 738.005410
epoch 8/50 | loss 613.125830
epoch 9/50 | loss 506.935382
epoch 10/50 | loss 410.126876
epoch 11/50 | loss 321.846054
epoch 12/50 | loss 251.143429
epoch 13/50 | loss 183.099998
epoch 14/50 | loss 133.026463
epoch 15/50 | loss 97.402689
epoch 16/50 | loss 72.702942
epoch 17/50 | loss 54.798548
epoch 18/50 | loss 41.550411
epoch 19/50 | loss 31.969172
epoch 20/50 | loss 24.786923
epoch 21/50 | loss 18.615787
epoch 22/50 | loss 15.032604
epoch 23/50 | loss 12.063268
epoch 24/50 | loss 10.009424
epoch 25/50 | loss 8.446872
epoch 26/50 | loss 7.194552
epoch 27/50 | loss 6.207230
epoch 28/50 | loss 5.405197
epoch 29/50 | loss 4.746954
epoch 30/50 | loss 4.225469
epoch 31/50 | loss 

In [ ]:
import numpy as np
import torch


# ---- config ---- #
SEQ_LEN = 30
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


# ---- load data ---- #
df = load_series("../assets/data/art_daily_flatmiddle.csv")

windows = load_windows(
    json_path="../assets/labels/combined_windows.json",
    file_key="artificialWithAnomaly/art_daily_flatmiddle.csv"
)

df = build_labels(df, windows)

values = df["value"].values.astype(np.float32)
labels = df["label"].values.astype(np.float32)


# ---- windowing (GROUND TRUTH SPACE) ---- #
x, y = make_windows_train(values, labels, SEQ_LEN)


# ---- stats ---- #
true_anomaly_ratio = np.mean(y == 1)
print(f"true_anomaly_ratio: {true_anomaly_ratio:.6f} ({true_anomaly_ratio*100:.2f}%)")
print(f"windows: {len(y)}")
print(f"anomaly_windows: {np.sum(y == 1)}")
print(f"normal_windows: {np.sum(y == 0)}")


# ---- load model ---- #
model = LSTMAutoencoder(input_dim=1, hidden_dim=128, latent_dim=32)
model.load_state_dict(torch.load("lstm_ae_model.pt", map_location=device))
model.to(device)
model.eval()


# ---- TRAIN distribution errors (NORMAL ONLY) ---- #
train_x = x[y == 0]
train_err, _ = run_inference(model, train_x, SEQ_LEN, device=device)


# ---- TEST (FULL SERIES - FIXED) ---- #
test_err, _ = run_inference(model, values, SEQ_LEN, device=device)


# ---- threshold ---- #
threshold = np.percentile(train_err, 73.01)


# ---- predictions ---- #
preds = (test_err > threshold).astype(np.int32)


# ---- output ---- #
print("threshold:", threshold)
print("train_error_mean:", train_err.mean())
print("test_error_mean:", test_err.mean())
print("anomalies:", preds.sum())
print("anomaly_ratio:", preds.mean())

device: cuda
true_anomaly_ratio: 0.107919 (10.79%)
windows: 4003
anomaly_windows: 432
normal_windows: 3571
threshold: 25.449406
train_error_mean: 4.522707
val_error_mean: 4.726796
test_error_mean: 4.536131
anomalies: 5733
anomaly_ratio: 0.04775072671392042
